# Build a SciFlawBench Task

Turn one row of the annotator spreadsheet into a **SciFlawBench task line**, and append it to a `.jsonl` task file.

**How to use it**

1. Run the *Setup* cell once.
2. Fill in the *Task fields* cell.
3. Run *Build and check* - it prints the JSONL line and every problem it found.
4. Run *Append to the task file* once the line is clean.
5. Run *Re-check the task file* before opening a pull request.

The rules enforced here are the ones in [`docs/task-authoring.md`](../docs/task-authoring.md) (fields, failure modes, tools) and [`docs/failure-modes.md`](../docs/failure-modes.md) (which mode to pick). The code behind this notebook lives in [`build_task.py`](build_task.py), and it needs the harness installed in this kernel (see the [README](../README.md)).

In [ ]:
import json
import sys
from pathlib import Path

from core.task_ids import find_repo_root

sys.path.insert(0, str(find_repo_root(Path.cwd()) / "utils"))

from build_task import (
    ALL_TOOLS,
    FAILURE_MODES,
    REPO_ROOT,
    VALIDATORS,
    append_task,
    build_task,
    check_task,
    check_task_file,
    display_path,
)

print("repo root:", REPO_ROOT)
print(f"{len(FAILURE_MODES)} failure modes, {len(VALIDATORS)} verifiers, {len(ALL_TOOLS)} tools")

repo root: /home/nicholas/Documents/SciFlawBench
11 failure modes, 9 verifiers, 8 tools


## Task Fields

Fill in every slot below. The values come straight off the submission: the **prompt** and **ground truth** from the sheet, the **failure-mode flags** you ticked, and the **tools** the item needs.

* `PROMPT` - the exact text handed to the agent. Keep it verbatim; by default the task id is the md5 of this text, so any edit changes the id. If the task reads a local file, write the literal `{TASK_ID}` placeholder in the path: the harness resolves it at run time, and the file must live in `data/tasks/assets/<task_id>/`.
* `TASK_ID` - leave it empty to use the md5 of the prompt as the id. Set a fixed id instead (it must start with a letter, e.g. `SFB-0042`) to keep the same id, and the same assets folder, when the prompt is edited later; the md5 is then stored in `prompt_md5`.
* `QUANT_FAILURE_MODES` / `QUAL_FAILURE_MODES` - the flags that are `true` for this item, as lists. At least one is required, and 1-3 is normal.
* `EXTRA_TOOLS` - only tools the default agents do **not** have. Listing a default tool is an error: smolagents rejects duplicate tool names.
* `VALIDATOR_DEFS` - the checks that turn the ground truth into a pass/fail. **The expected answer lives here**: a task file has no separate `ground_truth` field. Give **exactly one check per failure mode**, name that mode in the check's `failure_mode`, and list the checks in the order of the modes (quantitative first): each mode is scored from its own check.
* `SOURCES` - bookkeeping only: a list of `{'url': ..., 'note': ...}` objects, one per source, where `note` says what the source shows (e.g. the value found and the date it was consulted). It is stored in the task line as `sources` but never used by the harness or the evaluation.

In [ ]:
# EDIT ME

# File the task line is appended to (created if missing). Annotators can stage in their
# own file first, e.g. 'annotator-handle.jsonl', and merge into the official set after review.
from datetime import date

ANNOTATOR = ""  # your handle, e.g. "shivam"; "" keeps the default tasks.jsonl
OUTPUT_FILE = REPO_ROOT / "data" / "tasks" / "v0" / (
    f"{ANNOTATOR}-{date.today():%d%m%Y}.jsonl" if ANNOTATOR else "tasks.jsonl"
)
print("writing to:", display_path(OUTPUT_FILE))

# The exact prompt handed to the agent. Use triple quotes when it spans several lines.
PROMPT = ""

# "" uses md5(PROMPT) as the id; a fixed id such as "SFB-0042" stays the same when the prompt is edited.
TASK_ID = ""

AGENT_ID = "code_agent"  # code_agent | tool_agent

# The failure-mode flags that are true for this item (at least one; 1-3 is normal).
QUANT_FAILURE_MODES = []  # e.g. ['correctness']
QUAL_FAILURE_MODES = []  # e.g. ['sycophancy']

# Tools the default agents do NOT have. Never list a default tool: duplicate names crash the run.
EXTRA_TOOLS = [
    # {'tool_name': 'read_file'},
    # {'tool_name': 'json_answer_tool', 'kwargs': {'required_fields': ['papers']}},
]

# The expected answer: exactly one {'name': ..., 'kwargs': {...}, 'failure_mode': ...} check per failure mode,
# in the order of the modes above (quantitative first). See data/example/README.md for every verifier and its kwargs.
VALIDATOR_DEFS = [
    # {'name': 'content:numeric_match', 'kwargs': {'expected': '18.4', 'tol': 0.05}, 'failure_mode': 'correctness'},
    # {'name': 'judge:rubric', 'kwargs': {'rubric': 'Does the trace ...? Answer true only if ...'}, 'failure_mode': 'sycophancy'},
]

# Bookkeeping only, not used in evaluation: where the ground truth was verified.
SOURCES = []  # e.g. [{'url': 'https://arxiv.org/abs/2610.03455', 'note': 'abstract: bound stated in the first sentence'}]

In [ ]:
task = build_task(
    PROMPT, AGENT_ID, QUANT_FAILURE_MODES, QUAL_FAILURE_MODES, EXTRA_TOOLS, VALIDATOR_DEFS, SOURCES, task_id=TASK_ID
)
problems = check_task(task)
mode_count = len(task["failure_modes"]["quantitative"]) + len(task["failure_modes"]["qualitative"])

print("task_id:", task["task_id"])
print(json.dumps(task, ensure_ascii=False))
print()
if mode_count > 3:
    print(f"note: {mode_count} failure modes. Most items probe 1-3; consider splitting the item.")
if problems:
    print("Fix these before appending:")
    for problem in problems:
        print("  -", problem)
else:
    print("Looks good. Run the next cell to append it.")

task_id: d41d8cd98f00b204e9800998ecf8427e
{"task_id": "d41d8cd98f00b204e9800998ecf8427e", "task": "", "agent_id": "code_agent", "failure_modes": {"quantitative": [], "qualitative": []}, "extra_tools": [], "validators": []}

Fix these before appending:
  - the prompt is empty
  - declare at least one failure mode (quantitative or qualitative)
  - no validators: nothing checks the answer, so the task scores nothing


In [ ]:
if problems:
    print("Nothing was appended. Fix the problems above, then run this cell again.")
else:
    total = append_task(task, OUTPUT_FILE)
    print(f"appended task #{total} to {display_path(OUTPUT_FILE)}")

Nothing was appended. Fix the problems above, then run this cell again.


In [ ]:
# Re-check every task in OUTPUT_FILE with the same rules. Run this before a pull request.
rows, duplicates, problems = check_task_file(OUTPUT_FILE)

print(f"{display_path(OUTPUT_FILE)}: {len(rows)} task(s)")
print("duplicate ids:", duplicates or "none")
for problem in problems:
    print("  -", problem)
print("All tasks are valid." if not problems and not duplicates else "Problems found.")

data/tasks/v0/tasks.jsonl: 0 task(s)
duplicate ids: none
All tasks are valid.
